# 022 — Training: EfficientNet (deterministic + NLL, pre-Round-1 recipe)

Trains `efficientnet_unet` and `efficientnet_unet_nll` — the two architectures built on a pretrained EfficientNetB0 encoder. Consolidated here from three previous notebooks: `efficientnet_unet`'s deterministic training used to live in `020_training.ipynb` alongside `unet`/`resunet`/`attention_unet`; `efficientnet_unet_nll`'s `gaussian_nll`/`beta_nll` training used to live in `021_training_nll.ipynb`/`022_training_beta_nll.ipynb` alongside the other three NLL architectures. Both moved here because EfficientNet is **Round 2 territory** (`fixing.md`) — its pretrained encoder means the Round 1 recipe (`GroupNormalization`, `combined_loss` unification, single NLL loss) cannot be applied the same way as the from-scratch architectures without special handling, so this notebook still trains with the **pre-Round-1 recipe**: `combined_loss_advanced` for the deterministic model, and **both** `gaussian_nll`/`beta_nll` for the NLL model (not yet collapsed to one loss). See `020_training.ipynb`'s title cell for the full table of companion notebooks.

Only the **artwork-and-mockups** split is used (see §1).

Make the project root importable so `scripts.*` resolves regardless of
the notebook's working directory.

In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

Imports, a fixed global seed, and a GPU sanity check.

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf

from scripts.config import settings
from scripts.dataset import (
    build_dataset,
    load_image_pairs,
    mockup_aware_train_val_test_split,
)
from scripts.reproducibility import set_global_seed
from scripts.trainer import compile_model, get_callbacks, get_model
from scripts.trainer_nll import compile_model_nll, get_model_nll
from scripts.visualization import plot_training_curves

set_global_seed()

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")
print(f"TensorFlow version: {tf.__version__}")

## 1. Dataset — artwork-and-mockups split

Same split as `020`/`021`/`023` §1 — required so these checkpoints are
trained and evaluated under the same conditions as the rest of the
project.

In [ ]:
pairs = load_image_pairs(settings.IR_DIR, settings.RGB_DIR)
train_pairs, val_pairs, _ = mockup_aware_train_val_test_split(
    pairs,
    train_ratio=settings.TRAIN_RATIO,
    val_ratio=settings.VAL_RATIO,
    mockup_ids=settings.MOCKUP_ARTWORK_IDS,
    mockup_test_ratio=settings.MOCKUP_TEST_RATIO,
    seed=settings.SEED,
)

train_ds = build_dataset(
    train_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=True,
    shuffle=True,
    seed=settings.SEED,
    crop_size=settings.CROP_SIZE,
)
val_ds = build_dataset(
    val_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=False,
    shuffle=False,
)

print(f"Train: {len(train_pairs)} patches ({len(train_ds)} batches)")
print(f"Val:   {len(val_pairs)} patches ({len(val_ds)} batches)")

## 2. Loss functions

| Model | Loss |
|---|---|
| `efficientnet_unet` | `combined_loss_advanced` — MAE + Laplacian pyramid + FFT |
| `efficientnet_unet_nll` | `gaussian_nll_loss` **and** `beta_gaussian_nll_loss` (Seitzer et al. 2022) — trained twice, into separate checkpoints |

`efficientnet_unet`'s pretrained encoder reproduces high-frequency detail well, so it gets the loss that explicitly rewards it: the **Laplacian pyramid** term decomposes the error into spatial-frequency bands and weights finer detail more heavily; the **FFT** term penalises magnitude-spectrum error uniformly across all frequencies, preventing the model from trading away high-frequency accuracy for a lower low-frequency error. Both target the frequency band where underdrawing strokes live. `scripts.trainer.uses_advanced_loss` is the single source of truth for this — see `030_evaluation.ipynb`.

`efficientnet_unet_nll` predicts `(mu, log_var)` per pixel and is trained with both Gaussian NLL variants (unlike the other three NLL architectures, collapsed to one Laplace-beta loss in `021_training_nll.ipynb` — see `fixing.md` #10) so the `gaussian_nll` vs. `beta_nll` comparison stays available for this architecture pending Round 2.

The cell below visualises the pyramid decomposition and FFT spectra of one training sample, to make those two terms concrete before training starts.

In [ ]:
import numpy as np
from PIL import Image

# --- Laplacian pyramid on a sample IR image ---
ir_np = np.array(Image.open(train_pairs[0][1]).convert("L")).astype(np.float32) / 255.0
ir_t = tf.constant(ir_np[np.newaxis, ..., np.newaxis])  # (1, H, W, 1)


def _lap_level(x: tf.Tensor) -> tuple:
    low = tf.nn.avg_pool2d(x, ksize=2, strides=2, padding="VALID")
    up = tf.image.resize(low, tf.shape(x)[1:3], method="bilinear")
    detail = (x - up)[0, ..., 0].numpy()
    return detail, low


LEVELS = 5
details, x = [], ir_t
for _ in range(LEVELS):
    d, x = _lap_level(x)
    details.append(d)

fig, axes = plt.subplots(1, LEVELS + 1, figsize=(4 * (LEVELS + 1), 4))
fig.suptitle("Laplacian pyramid — sample IR image  (level 0 = finest detail)")
axes[0].imshow(ir_np, cmap="gray")
axes[0].set_title("Original IR")
axes[0].axis("off")
for i, d in enumerate(details):
    d_disp = (d - d.min()) / (d.max() - d.min() + 1e-8)
    axes[i + 1].imshow(d_disp, cmap="RdBu_r")
    axes[i + 1].set_title(f"Level {i}")
    axes[i + 1].axis("off")
plt.tight_layout()
plt.show()

# --- FFT magnitude spectra ---
rgb_np = (
    np.array(Image.open(train_pairs[0][0]).convert("RGB")).astype(np.float32) / 255.0
)
log_rgb = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(rgb_np[..., 0]))))
log_ir = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(ir_np))))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("FFT magnitude spectra (log scale) — same patch")
axes[0].imshow(log_rgb, cmap="inferno")
axes[0].set_title("RGB (channel R)")
axes[0].axis("off")
axes[1].imshow(log_ir, cmap="inferno")
axes[1].set_title("IR")
axes[1].axis("off")
axes[2].imshow(np.abs(log_rgb - log_ir), cmap="hot")
axes[2].set_title("Spectral difference |R − IR|")
axes[2].axis("off")
plt.tight_layout()
plt.show()

## 3. Train the deterministic model

`efficientnet_unet`, compiled with `combined_loss_advanced` (`compile_model`
derives it from `uses_advanced_loss`). Checkpoint goes to
`models/deterministic/efficientnet_unet/best_model.keras`. Set
`EPOCHS = 2` for a quick smoke test before committing to a full run.

> **Note:** downloads EfficientNetB0 ImageNet weights on first use.
> Subsequent runs use the local Keras cache.

In [ ]:
EPOCHS = settings.EPOCHS  # -- lower for a quick smoke test
DET_MODEL_DIR = settings.MODELS_DIR / "deterministic"
DET_LOG_DIR = settings.LOGS_DIR / "deterministic"

det_histories: dict = {}

arch = "efficientnet_unet"
print(f"\n{'=' * 60}")
print(f"  Architecture: {arch}")
print(f"{'=' * 60}")

model = get_model(arch)
model = compile_model(model, arch, lr=settings.LEARNING_RATE)
model.summary(line_length=80)

callbacks = get_callbacks(arch, log_dir=DET_LOG_DIR, model_dir=DET_MODEL_DIR)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)
det_histories[arch] = history.history

best_val_loss = min(history.history["val_loss"])
print(f"\nBest val_loss ({arch}): {best_val_loss:.4f}")

## 4. Train the NLL model — both loss variants

`efficientnet_unet_nll`, trained twice: once with `gaussian_nll`, once
with `beta_nll` (`BETA = settings.NLL_BETA`), into separate checkpoint
trees so neither overwrites the other.

In [ ]:
NLL_ARCH = "efficientnet_unet_nll"
NLL_LOSSES = {"gaussian_nll": "nll_gaussian", "beta_nll": "nll_beta"}

nll_histories: dict = {}

for loss_name, tree in NLL_LOSSES.items():
    print(f"\n{'=' * 60}")
    print(f"  Architecture: {NLL_ARCH}  (loss: {loss_name})")
    print(f"{'=' * 60}")

    nll_model_dir = settings.MODELS_DIR / tree
    nll_log_dir = settings.LOGS_DIR / tree

    model = get_model_nll(NLL_ARCH)
    model = compile_model_nll(
        model,
        lr=settings.LEARNING_RATE,
        loss_name=loss_name,
        beta=settings.NLL_BETA,
    )
    model.summary(line_length=80)

    callbacks = get_callbacks(NLL_ARCH, log_dir=nll_log_dir, model_dir=nll_model_dir)

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1,
    )
    nll_histories[loss_name] = history.history

    best_val_loss = min(history.history["val_loss"])
    print(f"\nBest val_loss ({NLL_ARCH}, {loss_name}): {best_val_loss:.4f}")

## 5. Training curves

In [ ]:
for arch, history in det_histories.items():
    plot_training_curves(history, title=f"Training history — {arch}")
    plt.show()

for loss_name, history in nll_histories.items():
    plot_training_curves(
        history, title=f"Training history — {NLL_ARCH} ({loss_name})"
    )
    plt.show()

## 6. Summary

Checkpoints saved to `models/deterministic/efficientnet_unet/best_model.keras`,
`models/nll_gaussian/efficientnet_unet_nll/best_model.keras`, and
`models/nll_beta/efficientnet_unet_nll/best_model.keras`. Logs under the
matching `logs/` subtrees.

In [ ]:
checks = [
    ("efficientnet_unet", DET_MODEL_DIR),
    ("efficientnet_unet_nll", settings.MODELS_DIR / "nll_gaussian"),
    ("efficientnet_unet_nll", settings.MODELS_DIR / "nll_beta"),
]
for arch, model_dir in checks:
    ckpt = model_dir / arch / "best_model.keras"
    status = "found" if ckpt.exists() else "MISSING"
    print(f"{arch:<25} ({model_dir.name:<12}): {status}  ({ckpt})")